# Logistic Regression via Autograd 

In [ ]:
from sklearn.datasets import load_breast_cancer
import torch

In [ ]:
device  = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
torch.manual_seed(0)

X, y = load_breast_cancer(as_frame=True, return_X_y=True)
X = torch.tensor(X.values, dtype=torch.float32, device=device)
y = torch.tensor(y.values, dtype=torch.float32, device=device)

In [ ]:
# Standardisation (par feature)
X = (X - X.mean(dim=0)) / X.std(dim=0)

num_feats = X.shape[1]

w = torch.randn(num_feats, requires_grad = True, device=device)
b = torch.randn(1, requires_grad = True, device=device)

# Modèle de logistic regression
def  sigmoid(x):
    return 1 / (1 + torch.exp(-x))

def compute_z(X, w, b):
    return sigmoid(torch.matmul(X, w) + b)

# Optimiseur et loss
learning_rate = 1e-1
optimizer = torch.optim.SGD([w, b], lr=learning_rate)
criterion = torch.nn.BCELoss()

# Entraînement
n_epochs = 500
for epoch in range(1, n_epochs+1):
    # Forward pass
    z = compute_z(X, w, b).clamp(min=0, max=1)

    # Affichage métrique
    accuracy = 100 * (((z > 0.5).float() == y).float()).mean().item()
    print(f'{accuracy=:.1f} %')

    # Calcul de la loss (erreur entre prédiction et vérité terrain)
    loss = criterion(z, y)

    # C'est ici que la magie opère -> Calcul du gradient automatique
    loss.backward()  # gradients

    # Màj de w, b par descente de gradient
    optimizer.step()  # w = w - learning_rate * gradients

    # Remise à zéro des gradients accumulés avant la prochaine itération (sinon ils s'accumulent)
    optimizer.zero_grad()


# Multi-Layer Perceptron

### Création de la classe MLP

In [ ]:
import torch
from torch import nn


class MLP(torch.nn.Module):
    def __init__(self, in_shape, n_neurons, num_hiddens_layers, num_classes):
        super(MLP, self).__init__()
        self.input_layer = nn.Linear(in_shape, n_neurons)
        layers = []
        for _ in range(num_hiddens_layers):
            layers.append(nn.Linear(n_neurons, n_neurons))
            layers.append(nn.ReLU())
        self.hidden_layers = nn.Sequential(*layers)
        self.output_layer = nn.Linear(n_neurons, num_classes, bias=False)

    def forward(self, x):
        i = nn.ReLU()(self.input_layer(x))
        h = self.hidden_layers(i)
        y = nn.Softmax(dim=1)(self.output_layer(h))

        return y


In [ ]:
from sklearn.datasets import fetch_openml
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

y = y.astype("int64")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axs = plt.subplots(5, 2)

fig.set_size_inches(12,7)
axs = axs.ravel()

for i in range(10):
    axs[i].imshow(X[np.argmax(y==i), :].reshape(28, 28))

In [ ]:
mnist_classifier = MLP(in_shape=X.shape[1], 
                       n_neurons=8, num_hiddens_layers=5, num_classes=10)

In [ ]:
sum(p.numel() for p in mnist_classifier.parameters())

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MNISTDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32) / 255. # + torch.randn(X.shape)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Retourne une paire (image, label)
        return self.X[idx], self.y[idx]

In [ ]:
m = X.shape[0]

idx_shuffle = torch.randperm(m)

X_train = X[idx_shuffle[:50000]]
X_val = X[idx_shuffle[50000:]]
y_train = y[idx_shuffle[:50000]]
y_val = y[idx_shuffle[50000:]]

In [ ]:
# Optimiseur et loss
learning_rate = 1e-3
optimizer = torch.optim.Adam(mnist_classifier.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss()  # BCELoss -> binary | MSELoss -> regression

batch_size = 32
train_dataset = MNISTDataset(X_train, y_train)
val_dataset = MNISTDataset(X_val, y_val)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Entraînement
mnist_classifier = mnist_classifier.to(device)
n_epochs = 100
for epoch in range(1, n_epochs+1):
    total_train_loss = 0
    total_val_loss = 0
    correct_train = 0
    correct_val = 0
    total_train = 0
    total_val = 0
    
    # --- Apprentissage ---
    mnist_classifier.train()
    for xb, yb in train_dataloader: # mini-batch
        xb = xb.to(device)
        yb = yb.to(device)
        # Forward pass
        y_pred = mnist_classifier(xb)

        # Loss
        loss = criterion(y_pred, yb)

        # Backward
        loss.backward()  # gradient
        optimizer.step() # application de w <- w - learning_rate * gradients
        optimizer.zero_grad()  # màz du gradient
        
        # Accumulation des pertes
        total_train_loss += loss.item() * xb.size(0)

        # Accuracy
        preds = torch.argmax(y_pred, dim=1)
        correct_train += (preds == yb).sum().item()
        total_train += yb.size(0)

    epoch_loss_train = total_train_loss / total_train
    epoch_acc_train = 100 * correct_train / total_train

    # --- Validation ---
    mnist_classifier.eval()
    with torch.no_grad():
        for xb, yb in val_dataloader:
            xb = xb.to(device)
            yb = yb.to(device)
            y_pred = mnist_classifier(xb)
            loss = criterion(y_pred, yb)

            total_val_loss += loss.item() * xb.size(0)

            preds = torch.argmax(y_pred, dim=1)
            correct_val += (preds == yb).sum().item()
            total_val += yb.size(0)

    epoch_loss_val = total_val_loss / total_val
    epoch_acc_val = 100 * correct_val / total_val

    print(f"Epoch {epoch:3d} | "
          f"Train Loss: {epoch_loss_train:.4f} | Train Acc: {epoch_acc_train:.2f}% | "
          f"Val Loss: {epoch_loss_val:.4f} | Val Acc: {epoch_acc_val:.2f}%")


In [ ]:
print(mnist_classifier.hidden_layers[0].weight.grad)

In [ ]:
idx = np.random.randint(m)
pred_class = torch.argmax(mnist_classifier(torch.tensor(X[idx]).view(1, -1).to(device).to(torch.float) / 255.)).item()
plt.imshow(X[idx].reshape(28, 28))
plt.title(f'Prediction={pred_class}')

# Exercice 
Réécrire la première cellule avec la logique PyTorch.

Cela nécessite de :
* Créer un dataset et un dataloader
* Créer un modèle LogisticRegression en héritant de torch.nn.Module

## MLP sur california housing dataset